In [3]:
import os
import wfdb
import numpy as np
import neurokit2 as nk
from scipy.signal import butter, filtfilt
from tqdm import tqdm

# =========================
# PARAMETERS
# =========================
DATASET_PATH = "/Users/riasnair/Downloads/CAPSTONE_TEAM_104/mit-bih-arrhythmia-database-1.0.0"

WINDOW_SEC = 0.6          # seconds (P-QRS-T)
EPS = 1e-8

CLASS_MAP = {
    "N": 0,   # Normal
    "V": 1,   # PVC
    "A": 2,   # APC
    "L": 3,   # LBBB
    "R": 4    # RBBB
}

# =========================
# BANDPASS FILTER
# =========================
def bandpass_filter(ecg, fs, low=0.5, high=45):
    nyq = 0.5 * fs
    b, a = butter(2, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, ecg)

# =========================
# STORAGE
# =========================
X = []
y = []
record_ids = []   # for patient-wise split later

# =========================
# LOAD RECORD LIST
# =========================
records = wfdb.get_record_list("mitdb")
print("Total records:", len(records))

# =========================
# MAIN LOOP
# =========================
for record in tqdm(records, desc="Processing records"):
    record_path = os.path.join(DATASET_PATH, record)

    signal, fields = wfdb.rdsamp(record_path)
    annotation = wfdb.rdann(record_path, "atr")

    fs = fields["fs"]
    ecg = signal[:, 0]

    # Window parameters
    WINDOW = int(WINDOW_SEC * fs)
    HALF = WINDOW // 2

    # -------------------------
    # CLEAN ECG
    # -------------------------
    ecg_filtered = bandpass_filter(ecg, fs)

    # -------------------------
    # R-PEAK DETECTION
    # -------------------------
    _, info = nk.ecg_process(ecg_filtered, sampling_rate=fs)
    r_peaks = info["ECG_R_Peaks"]

    # -------------------------
    # BEAT SEGMENTATION
    # -------------------------
    for r in r_peaks:
        if r - HALF < 0 or r + HALF >= len(ecg_filtered):
            continue

        beat = ecg_filtered[r - HALF : r + HALF]

        # ---- Nearest annotation (CRITICAL FIX)
        ann_idx = np.argmin(np.abs(annotation.sample - r))
        symbol = annotation.symbol[ann_idx]

        if symbol not in CLASS_MAP:
            continue

        # -------------------------
        # CLIP EXTREMES
        # -------------------------
        std = np.std(beat)
        beat = np.clip(beat, -3 * std, 3 * std)

        # -------------------------
        # NORMALIZE
        # -------------------------
        beat = (beat - np.mean(beat)) / (std + EPS)

        # -------------------------
        # MULTI-CHANNEL FEATURES
        # -------------------------
        d1 = np.diff(beat, prepend=beat[0])   # first derivative
        d2 = np.diff(d1, prepend=d1[0])       # second derivative

        beat_multi = np.stack([beat, d1, d2], axis=-1)

        # -------------------------
        # STORE
        # -------------------------
        X.append(beat_multi)
        y.append(CLASS_MAP[symbol])
        record_ids.append(record)

# =========================
# FINAL ARRAYS
# =========================
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
record_ids = np.array(record_ids)

print("\n========== DATASET READY ==========")
print("X shape:", X.shape)     # (samples, time_steps, channels)
print("y shape:", y.shape)
print("Classes:", np.unique(y, return_counts=True))


Total records: 48


Processing records: 100%|███████████████████████| 48/48 [08:23<00:00, 10.48s/it]



========== DATASET READY ==========
X shape: (100782, 216, 3)
y shape: (100782,)
Classes: (array([0, 1, 2, 3, 4]), array([76357,  6895,  2461,  7775,  7294]))
